# 05. Два представления одного набора данных — разбор

Здесь нет обучения модели. Мы сравним представление для управленческого отчёта и для будущего сравнения состояний. Используется **фиксированная стартовая версия**, чтобы результат не зависел от ошибок предыдущей лабораторной.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
DATA=Path('../05_ДАННЫЕ_СТУДЕНТАМ')
measure=pd.read_csv(DATA/'campus_readings_TOPIC05_START.csv')
rooms=pd.read_csv(DATA/'rooms_catalog.csv')
messages=pd.read_csv(DATA/'campus_messages.csv')
print(len(measure),'наблюдений;',len(rooms),'помещений;',len(messages),'сообщений')
measure.head(3)

## 1. Сначала смысловой тип

Назовите четыре разных типа: числовое измерение, категориальная метка, технический идентификатор, время. На будущее: числовые суффиксы `record_id` не станут полезнее от применения StandardScaler.

In [ ]:
assert rooms.room_id.is_unique
joined=measure.merge(rooms,left_on='room_code',right_on='room_id',how='left',validate='many_to_one')
assert len(joined)==len(measure)
print('Несопоставленных помещений:',joined.room_id.isna().sum())

## 2. Производные признаки под конкретную цель

Доля заполнения позволяет сопоставлять аудитории разной вместимости. Новые поля должны иметь понятные формулы и ограничения.

In [ ]:
joined['time']=pd.to_datetime(joined['observed_at'])
joined['date']=joined['time'].dt.date
joined['hour']=joined['time'].dt.hour
joined['occupancy_ratio']=joined['people_count'] / joined['capacity']
assert (joined['capacity']>0).all()
print('18 человек / 20 мест =',18/20, '; 18 / 60 =',18/60)
joined[['record_id','room_code','people_count','capacity','occupancy_ratio','hour']].head()

## 3. Представление А — для человека

Сохраняем понятные единицы, названия и флаги качества. Обозначьте, что это учебные данные, а не официальная оценка состояния помещений.

In [ ]:
A=joined[['record_id','room_code','building','room_type','date','hour','co2_ppm','temp_c','occupancy_ratio','quality_flag']].copy()
A.head(5)

## 4. Представление Б — численные признаки для сравнения

Для демонстрации используем фиксированные учебные шкалы, явно записанные рядом с признаками. Здесь это *не* нормативы воздуха или температуры, а способ показать разницу числовых масштабов. Не делайте вид, будто `room_type=1,2,3` — осмысленная шкала расстояния.

In [ ]:
PARAMS={'co2_min':400,'co2_max':1600,'temp_min':16,'temp_max':30}
B=joined[['record_id','quality_flag']].copy()
B['co2_scaled']=(joined['co2_ppm']-PARAMS['co2_min'])/(PARAMS['co2_max']-PARAMS['co2_min'])
B['temp_scaled']=(joined['temp_c']-PARAMS['temp_min'])/(PARAMS['temp_max']-PARAMS['temp_min'])
B['occupancy_ratio']=joined['occupancy_ratio']
encoded=pd.get_dummies(joined['room_type'],prefix='type',dtype=int)
B=pd.concat([B,encoded],axis=1)
print('Параметры:',PARAMS)
B.head(3)

**Внимание:** при `CO₂ = 4200 ppm` значение `co2_scaled` будет больше 1. Это не ошибка формулы. Обдумайте, допускается ли такое наблюдение в задаче сравнения и почему его нельзя молча обрезать. В собственном решении можете обоснованно выбрать другое масштабирование.

In [ ]:
r=B.loc[joined['co2_ppm']>2000,['record_id','co2_scaled','quality_flag']]
print('Нетипичные строки в фиксированной шкале:')
print(r.to_string(index=False))
assert B['record_id'].nunique()==len(joined)
assert A['record_id'].nunique()==len(joined)

## 5. Схема должна работать и завтра

Фиксированный словарь категорий и параметры преобразования должны применяться ко всем последующим партиям; иначе одно и то же состояние может получить разные координаты. Для неизвестной категории нужно описать fallback, а не раздвигать столбцы незаметно.

In [ ]:
NEW_CO2=1000
fixed_scaled=(NEW_CO2-PARAMS['co2_min'])/(PARAMS['co2_max']-PARAMS['co2_min'])
print('1000 ppm в фиксированной шкале =',round(fixed_scaled,3))
assert abs(fixed_scaled-0.5)<1e-12

## 6. Как извлечь признаки из текста — и проверить их

Задача с сообщениями отличается от арифметики. Ниже — **не LLM**, а примитивное правило как контрольная база. Оно намеренно допускает неточности; сравните результаты с ручной разметкой 8–10 сообщений. Если доступна LLM, проверьте её по той же схеме.

In [ ]:
def baseline_topic(message):
    m=str(message).lower()
    if 'датчик' in m:return 'датчик'
    if 'дух' in m or 'вентил' in m or 'проветр' in m:return 'воздух'
    if 'группа' in m:return 'расписание'
    return 'нужна проверка'
messages['topic_baseline']=messages['message'].map(baseline_topic)
print(messages[['message_id','message','topic_baseline']].head(10).to_string(index=False))

## 7. Три итоговые проверки

- `record_id` одинаковы в А и Б, ни одно наблюдение не пропущено;
- значение `occupancy_ratio` проверено вручную не менее чем в пяти строках;
- ни один признак не кодирует знание, которое появится только после будущего решения.

В собственной лабораторной добавьте контроль нового неизвестного типа аудитории и случай отсутствующей вместимости. Обязательные результаты перечислены в HTML-задании.

In [ ]:
assert set(A.record_id)==set(B.record_id)
assert joined.loc[joined.capacity==20,'occupancy_ratio'].notna().all()
print('Демонстрационные проверки завершены')